# Processing Boundary Geometries for the Frontend

This notebook turns the Statistics Canada boundary shapefiles in `data/final_geoms/` into
**production GeoJSON** a SvelteKit map component can `fetch()` directly - no shapefile
parsing, no CRS handling, on the client.

**Sources** (all `data/final_geoms/`, all NAD83 / StatCan Lambert (EPSG:3347) except the
`toronto`/`vancouver` 2km buffer files, which are already WGS84):
- `provinces_territories/{ab,bc,...}.shp` - 13 separate province/territory shapefiles
- `csd_gtha/csds_metro_gtha.shp` - 27 census subdivisions covering the GTHA (one is the actual City of Toronto)
- `csd_vancouver/csds_metro_van.shp` - 37 census subdivisions covering Metro Vancouver (one is the actual City of Vancouver)
- `cma_gtha/toronto_hamilton_cma.shp`, `cma_vancouver/vancouver_cma.shp` - single-feature CMA outlines
- `toronto/toronto_merged_2km.shp`, `vancouver/vancouver_merged_2km.shp` - match-day 2km buffer zones (WGS84 already)

**Output format decisions** (confirmed before writing any code):
- Standard RFC 7946 GeoJSON `FeatureCollection`s, reprojected to **EPSG:4326** (lon/lat) - the
  GeoJSON default CRS, so no CRS member is written.
- Coordinates rounded to **5 decimal places** (~1.1m precision) - shrinks file size losslessly
  for display purposes.
- Only the property fields the frontend actually needs are kept (e.g. `CSDNAME`/`CSDUID`,
  `CMANAME`, `PRENAME`/`PREABBR`/`PRUID`) - `LANDAREA`, `Shape_Leng`, `Shape_Area`, `DGUID`, etc.
  are all dropped.
- **Simplification** is a rudimentary, low-risk pass only, applied in the *projected* Lambert
  CRS (meters, so tolerances are literal distances) before reprojecting to WGS84, with
  `preserve_topology=True`. It's skipped for files that are already small, and used more
  aggressively for country-scale layers than city-scale ones (see Section 2 for exact tolerances
  and the one exception found along the way).
- Missing values are never written - `json.dump(..., allow_nan=False)` is used as a hard
  guardrail, matching `process_activity_and_origins.ipynb`.

**Output location**: `src/data/geo/` - fetched directly by the SvelteKit frontend:
```
src/data/geo/
  zone_2km/
    toronto.json, vancouver.json
  csd/
    toronto.json, vancouver.json, gtha.json, metro_vancouver.json
  cma/
    toronto_hamilton.json, vancouver.json
  provinces_territories.json
```


In [1]:
import json
from pathlib import Path

import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm


In [2]:
GEOMS_DIR = Path("../../data/final_geoms")

# Output folder - fetched directly by the SvelteKit frontend
OUTPUT_DIR = Path("../../src/data/geo")
ZONE_2KM_DIR = OUTPUT_DIR / "zone_2km"
CSD_DIR = OUTPUT_DIR / "csd"
CMA_DIR = OUTPUT_DIR / "cma"
for d in [ZONE_2KM_DIR, CSD_DIR, CMA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Rounding applied just before writing JSON - ~1.1m precision, well below anything visible on a map
ROUND_COORD = 5

# Simplification tolerances (meters, applied in the source Lambert CRS before reprojecting).
# Kept deliberately conservative - a rudimentary pass, not meant to be visually noticeable.
TOL_CSD = 25         # city/neighbourhood scale (CSD-level files)
TOL_CMA = 60         # metro scale (CMA outlines)
TOL_PROVINCE = 200   # country scale (provinces/territories background layer)

PROVINCE_ABBRS = ["ab", "bc", "mb", "nb", "nl", "ns", "nt", "nu", "on", "pe", "qb", "sk", "yk"]


## 1. Helpers

Four small utilities shared by every section below:
- `fix_invalid` - `buffer(0)`-repairs only the geometries that actually fail `.is_valid`
  (checked before and after simplification, since simplifying can occasionally nick a polygon
  invalid even when the source was clean).
- `fix_winding` - re-orients every ring to the RFC 7946 winding convention (counterclockwise
  exterior rings, clockwise holes). Esri shapefiles follow the *opposite* (clockwise-exterior)
  convention, and while that doesn't matter for planar/SVG rendering (fill-rule doesn't care
  about winding direction), it breaks any consumer that treats the ring as a path on the
  sphere - `d3-geo`'s `fitExtent`/`geoBounds` in particular will interpret a clockwise-wound
  polygon as covering the *complement* (the rest of the world minus that shape) instead of the
  shape itself. Found this by fitting a projection to the Toronto CSD polygon and seeing it
  render as a full-world extent instead of a ~40km-wide city.
- `round_coords` - recursively rounds every coordinate in a GeoJSON coordinate array, since
  neither `GeoDataFrame.to_file` nor `.to_json` support rounding directly.
- `write_geojson` - keeps only the requested property columns, fixes ring winding, rounds
  coordinates, and writes compact (not pretty-printed) JSON - this is fetched by a web frontend,
  not read by a human.

In [3]:
from shapely.geometry import MultiPolygon
from shapely.geometry.polygon import orient


def fix_invalid(gdf):
    """Buffer(0)-fix only the geometries that are actually invalid. Returns (gdf, n_fixed)."""
    invalid = ~gdf.geometry.is_valid
    n_invalid = int(invalid.sum())
    if n_invalid:
        gdf.loc[invalid, gdf.geometry.name] = gdf.loc[invalid, gdf.geometry.name].buffer(0)
    return gdf, n_invalid


def fix_winding(gdf):
    """Re-orient every ring so d3-geo (the frontend's projection library) treats it as a small
    local shape rather than "the rest of the world". d3-geo's spherical winding convention
    turns out to be the *opposite* of the standard planar/shoelace convention (verified
    empirically: `shapely.orient(sign=1.0)`, the textbook "RFC 7946 counterclockwise exterior"
    orientation, made every single feature register as whole-world-sized under
    `d3.geoBounds`; `sign=-1.0` fixes all of them) - so `sign=-1.0` is what's actually correct
    for this frontend, despite looking backwards on paper."""
    def reorient(geom):
        if geom.geom_type == "Polygon":
            return orient(geom, sign=-1.0)
        elif geom.geom_type == "MultiPolygon":
            return MultiPolygon([orient(p, sign=-1.0) for p in geom.geoms])
        return geom
    gdf = gdf.copy()
    gdf[gdf.geometry.name] = gdf.geometry.apply(reorient)
    return gdf


def round_coords(coords, ndigits=ROUND_COORD):
    """Recursively round every float coordinate in a GeoJSON coordinate array."""
    if coords and isinstance(coords[0], (int, float)):
        return [round(v, ndigits) for v in coords]
    return [round_coords(c, ndigits) for c in coords]


def write_geojson(gdf, out_path, keep_cols):
    """Assumes gdf is already in EPSG:4326. Keeps only keep_cols + geometry, fixes ring
    winding, rounds coordinates, and writes compact JSON (allow_nan=False as a hard guardrail)."""
    gdf = fix_winding(gdf[keep_cols + [gdf.geometry.name]])
    fc = json.loads(gdf.to_json())
    for feat in fc["features"]:
        feat["geometry"]["coordinates"] = round_coords(feat["geometry"]["coordinates"])
    with open(out_path, "w") as f:
        json.dump(fc, f, allow_nan=False, separators=(",", ":"))
    return fc

## 2. Match-day 2km buffer zones (`zone_2km/`)

`toronto_merged_2km.shp` and `vancouver_merged_2km.shp` are already WGS84, so no reprojection
or simplification is needed - they're small already. Two things need cleaning up:
- Toronto's file has a stray `Z=0` third dimension on every vertex (`POLYGON Z`) - dropped with
  `force_2d()` so the output is plain 2D GeoJSON.
- Vancouver's file has **2 separate feature rows** (not dissolved), so they're unioned into one
  (Multi)Polygon feature before export.


In [4]:
tor_2km = gpd.read_file(GEOMS_DIR / "toronto/toronto_merged_2km.shp")
tor_2km = tor_2km.set_geometry(tor_2km.geometry.force_2d())  # drop the stray Z=0 dimension
tor_2km, n_inv = fix_invalid(tor_2km)
tor_2km["id"] = tor_2km.get("Id", 0)
print(f"toronto_merged_2km: {len(tor_2km)} feature(s), crs={tor_2km.crs}, invalid fixed={n_inv}")
write_geojson(tor_2km, ZONE_2KM_DIR / "toronto.json", keep_cols=["id"])

van_2km = gpd.read_file(GEOMS_DIR / "vancouver/vancouver_merged_2km.shp")
van_2km = van_2km.set_geometry(van_2km.geometry.force_2d())
van_2km, n_inv = fix_invalid(van_2km)
print(f"vancouver_merged_2km: {len(van_2km)} feature(s) before dissolve, invalid fixed={n_inv}")

dissolved_geom = van_2km.union_all() if hasattr(van_2km, "union_all") else van_2km.unary_union
van_2km_dissolved = gpd.GeoDataFrame({"id": [0]}, geometry=[dissolved_geom], crs=van_2km.crs)
van_2km_dissolved, n_inv = fix_invalid(van_2km_dissolved)
print(f"vancouver_merged_2km: dissolved to {len(van_2km_dissolved)} feature, "
      f"geom_type={van_2km_dissolved.geometry.iloc[0].geom_type}, invalid fixed={n_inv}")
write_geojson(van_2km_dissolved, ZONE_2KM_DIR / "vancouver.json", keep_cols=["id"])


toronto_merged_2km: 1 feature(s), crs=EPSG:4326, invalid fixed=0
vancouver_merged_2km: 2 feature(s) before dissolve, invalid fixed=0
vancouver_merged_2km: dissolved to 1 feature, geom_type=MultiPolygon, invalid fixed=0


{'type': 'FeatureCollection',
 'features': [{'id': '0',
   'type': 'Feature',
   'properties': {'id': 0},
   'geometry': {'type': 'MultiPolygon',
    'coordinates': [[[[-123.11219, 49.25759],
       [-123.11399, 49.25763],
       [-123.1153, 49.2577],
       [-123.11586, 49.25774],
       [-123.11633, 49.25778],
       [-123.11811, 49.25797],
       [-123.11986, 49.25824],
       [-123.12158, 49.25858],
       [-123.12326, 49.259],
       [-123.1249, 49.25948],
       [-123.12649, 49.26004],
       [-123.12751, 49.26044],
       [-123.12773, 49.26054],
       [-123.12796, 49.26063],
       [-123.12866, 49.26092],
       [-123.13012, 49.26161],
       [-123.13151, 49.26236],
       [-123.13282, 49.26316],
       [-123.13393, 49.26393],
       [-123.13431, 49.26421],
       [-123.13443, 49.2643],
       [-123.13557, 49.26521],
       [-123.13662, 49.26616],
       [-123.13757, 49.26716],
       [-123.13842, 49.2682],
       [-123.13916, 49.26927],
       [-123.1398, 49.27037],
       [-1

## 3. Single-city CSD boundaries (`csd/toronto.json`, `csd/vancouver.json`)

The real City of Toronto / City of Vancouver polygons, pulled out of the GTHA / Metro Vancouver
CSD collections by name (`toronto_csd.shp` in the source data is broken - 0 features - so this
filter is the correct source instead) and reprojected to WGS84.

**Judgement call**: the brief for this notebook assumed these single-city extracts would be
small enough to skip simplification entirely. That holds for Vancouver (3,247 vertices, ~69KB),
but **not** for Toronto - its raw CSD boundary carries ~116,000 vertices (very fine shoreline
and harbour-island detail), producing a 2.3MB file for a single polygon, larger than the
*entire* 27-feature simplified GTHA collection. Since simplifying it at the same tolerance used
for the CSD-level files (`TOL_CSD` = 25m) changes its area by under 0.02% - imperceptible - and
cuts the file down to ~25KB, that tolerance is applied to Toronto only; Vancouver is left as-is.


In [5]:
csd_gtha = gpd.read_file(GEOMS_DIR / "csd_gtha/csds_metro_gtha.shp")
csd_van = gpd.read_file(GEOMS_DIR / "csd_vancouver/csds_metro_van.shp")

tor_csd = csd_gtha[csd_gtha["CSDNAME"] == "Toronto"].copy()
assert len(tor_csd) == 1
tor_csd, n_inv = fix_invalid(tor_csd)
tor_csd["geometry"] = tor_csd.geometry.simplify(TOL_CSD, preserve_topology=True)  # see note above
tor_csd, n_inv2 = fix_invalid(tor_csd)
tor_csd = tor_csd.to_crs(4326)
print(f"csd/toronto: invalid fixed pre/post-simplify={n_inv}/{n_inv2}, bounds={tor_csd.total_bounds}")
write_geojson(tor_csd, CSD_DIR / "toronto.json", keep_cols=["CSDNAME", "CSDUID"])

van_csd = csd_van[csd_van["CSDNAME"] == "Vancouver"].copy()
assert len(van_csd) == 1
van_csd, n_inv = fix_invalid(van_csd)  # small already (~3.2k vertices) - no simplification needed
van_csd = van_csd.to_crs(4326)
print(f"csd/vancouver: invalid fixed={n_inv}, bounds={van_csd.total_bounds}")
write_geojson(van_csd, CSD_DIR / "vancouver.json", keep_cols=["CSDNAME", "CSDUID"])


csd/toronto: invalid fixed pre/post-simplify=0/0, bounds=[-79.63930241  43.5792011  -79.11533294  43.8554655 ]
csd/vancouver: invalid fixed=0, bounds=[-123.22489916   49.19985705 -123.02293653   49.31407865]


{'type': 'FeatureCollection',
 'features': [{'id': '7',
   'type': 'Feature',
   'properties': {'CSDNAME': 'Vancouver', 'CSDUID': '5915022'},
   'geometry': {'type': 'MultiPolygon',
    'coordinates': [[[[-123.17852, 49.21636],
       [-123.17854, 49.21624],
       [-123.17876, 49.2162],
       [-123.17905, 49.21619],
       [-123.17932, 49.2162],
       [-123.17954, 49.21624],
       [-123.17977, 49.21629],
       [-123.18002, 49.21633],
       [-123.18029, 49.21636],
       [-123.18054, 49.21639],
       [-123.1808, 49.21642],
       [-123.18105, 49.21645],
       [-123.18128, 49.21649],
       [-123.1815, 49.21653],
       [-123.18175, 49.21656],
       [-123.18197, 49.21661],
       [-123.18215, 49.21667],
       [-123.18237, 49.21672],
       [-123.18256, 49.21677],
       [-123.1828, 49.2169],
       [-123.18296, 49.21698],
       [-123.18326, 49.21716],
       [-123.18343, 49.21725],
       [-123.18359, 49.21732],
       [-123.18379, 49.21739],
       [-123.18399, 49.21746],
   

## 4. Full CSD collections (`csd/gtha.json`, `csd/metro_vancouver.json`)

All 27 GTHA / 37 Metro Vancouver census-subdivision polygons, reprojected to WGS84 with light
simplification (`TOL_CSD` = 25m - this is the largest/most detailed scale, so the smallest
tolerance). `CSDUID` is kept as the unique key rather than deduping by `CSDNAME` - a couple of
names repeat (e.g. two "Langley", two "North Vancouver" - a city and a district/township
sharing a name).


In [6]:
gtha_all = csd_gtha.copy()
gtha_all, n_inv = fix_invalid(gtha_all)
gtha_all["geometry"] = gtha_all.geometry.simplify(TOL_CSD, preserve_topology=True)
gtha_all, n_inv2 = fix_invalid(gtha_all)
gtha_all = gtha_all.to_crs(4326)
print(f"csd/gtha: {len(gtha_all)} features, invalid fixed pre/post-simplify={n_inv}/{n_inv2}")
write_geojson(gtha_all, CSD_DIR / "gtha.json", keep_cols=["CSDNAME", "CSDUID"])

van_all = csd_van.copy()
van_all, n_inv = fix_invalid(van_all)
van_all["geometry"] = van_all.geometry.simplify(TOL_CSD, preserve_topology=True)
van_all, n_inv2 = fix_invalid(van_all)
van_all = van_all.to_crs(4326)
print(f"csd/metro_vancouver: {len(van_all)} features, invalid fixed pre/post-simplify={n_inv}/{n_inv2}")
write_geojson(van_all, CSD_DIR / "metro_vancouver.json", keep_cols=["CSDNAME", "CSDUID"])


csd/gtha: 27 features, invalid fixed pre/post-simplify=0/0
csd/metro_vancouver: 37 features, invalid fixed pre/post-simplify=0/0


{'type': 'FeatureCollection',
 'features': [{'id': '0',
   'type': 'Feature',
   'properties': {'CSDNAME': 'Langley', 'CSDUID': '5915001'},
   'geometry': {'type': 'Polygon',
    'coordinates': [[[-122.65965, 49.19763],
      [-122.64233, 49.20458],
      [-122.63648, 49.20668],
      [-122.62837, 49.20888],
      [-122.6238, 49.2098],
      [-122.61777, 49.21045],
      [-122.60823, 49.21025],
      [-122.60155, 49.20853],
      [-122.59178, 49.20185],
      [-122.58946, 49.19945],
      [-122.58472, 49.19128],
      [-122.58251, 49.18896],
      [-122.57732, 49.18526],
      [-122.57379, 49.18367],
      [-122.55321, 49.17788],
      [-122.54818, 49.17562],
      [-122.54265, 49.17179],
      [-122.53846, 49.17015],
      [-122.52924, 49.16811],
      [-122.52523, 49.16758],
      [-122.51345, 49.16723],
      [-122.48878, 49.16833],
      [-122.47819, 49.1698],
      [-122.46717, 49.17046],
      [-122.46148, 49.17133],
      [-122.4614, 49.15079],
      [-122.46045, 49.12749],
    

## 5. CMA outlines (`cma/toronto_hamilton.json`, `cma/vancouver.json`)

Single-feature Census Metropolitan Area boundaries, reprojected to WGS84 with a slightly
larger tolerance than the CSD files (`TOL_CMA` = 60m - metro scale, less detail needed than
city/neighbourhood scale). Only `CMANAME` is kept.

Note: the Toronto-Hamilton CMA's `CMANAME` field value is literally `"Hamilton"` in the source
data (StatCan's official name for CMA 537) - not a bug, just worth flagging since the file is
saved as `toronto_hamilton.json`.


In [7]:
cma_gtha = gpd.read_file(GEOMS_DIR / "cma_gtha/toronto_hamilton_cma.shp")
cma_gtha, n_inv = fix_invalid(cma_gtha)
cma_gtha["geometry"] = cma_gtha.geometry.simplify(TOL_CMA, preserve_topology=True)
cma_gtha, n_inv2 = fix_invalid(cma_gtha)
cma_gtha = cma_gtha.to_crs(4326)
print(f"cma/toronto_hamilton: CMANAME={cma_gtha['CMANAME'].iloc[0]!r}, invalid fixed pre/post-simplify={n_inv}/{n_inv2}")
write_geojson(cma_gtha, CMA_DIR / "toronto_hamilton.json", keep_cols=["CMANAME"])

cma_van = gpd.read_file(GEOMS_DIR / "cma_vancouver/vancouver_cma.shp")
cma_van, n_inv = fix_invalid(cma_van)
cma_van["geometry"] = cma_van.geometry.simplify(TOL_CMA, preserve_topology=True)
cma_van, n_inv2 = fix_invalid(cma_van)
cma_van = cma_van.to_crs(4326)
print(f"cma/vancouver: CMANAME={cma_van['CMANAME'].iloc[0]!r}, invalid fixed pre/post-simplify={n_inv}/{n_inv2}")
write_geojson(cma_van, CMA_DIR / "vancouver.json", keep_cols=["CMANAME"])


cma/toronto_hamilton: CMANAME='Hamilton', invalid fixed pre/post-simplify=0/0
cma/vancouver: CMANAME='Vancouver', invalid fixed pre/post-simplify=0/0


{'type': 'FeatureCollection',
 'features': [{'id': '0',
   'type': 'Feature',
   'properties': {'CMANAME': 'Vancouver'},
   'geometry': {'type': 'MultiPolygon',
    'coordinates': [[[[-123.23999, 49.44407],
       [-123.24003, 49.44393],
       [-123.24008, 49.4441],
       [-123.23999, 49.44407]]],
     [[[-123.37862, 49.40264],
       [-123.37873, 49.40245],
       [-123.37882, 49.40254],
       [-123.37862, 49.40264]]],
     [[[-123.32826, 49.38266],
       [-123.32834, 49.38236],
       [-123.32839, 49.38259],
       [-123.32826, 49.38266]]],
     [[[-123.31206, 49.41601],
       [-123.31236, 49.41577],
       [-123.31231, 49.41602],
       [-123.31206, 49.41601]]],
     [[[-123.27474, 49.37693],
       [-123.27485, 49.37661],
       [-123.2749, 49.37692],
       [-123.27474, 49.37693]]],
     [[[-122.92558, 49.32352],
       [-122.92591, 49.32325],
       [-122.92599, 49.32337],
       [-122.92558, 49.32352]]],
     [[[-122.92828, 49.3205],
       [-122.92788, 49.32048],
       [-

## 6. Provinces & territories (`provinces_territories.json`)

All 13 separate province/territory shapefiles concatenated into one FeatureCollection,
reprojected to WGS84. This is the country-scale background layer, so the most aggressive
tolerance is used (`TOL_PROVINCE` = 200m - coastline detail this fine is imperceptible at that
zoom). Only `PRENAME`, `PREABBR`, `PRUID` are kept (`LANDAREA`/`Shape_Leng`/`Shape_Area`/`DGUID`
dropped).


In [8]:
frames = [gpd.read_file(GEOMS_DIR / f"provinces_territories/{abbr}.shp")
          for abbr in tqdm(PROVINCE_ABBRS, desc="loading province shapefiles")]

provinces = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=frames[0].crs)
provinces, n_inv = fix_invalid(provinces)
provinces["geometry"] = provinces.geometry.simplify(TOL_PROVINCE, preserve_topology=True)
provinces, n_inv2 = fix_invalid(provinces)
provinces = provinces.to_crs(4326)
print(f"provinces_territories: {len(provinces)} features, invalid fixed pre/post-simplify={n_inv}/{n_inv2}")
write_geojson(provinces, OUTPUT_DIR / "provinces_territories.json", keep_cols=["PRENAME", "PREABBR", "PRUID"])


loading province shapefiles:   0%|          | 0/13 [00:00<?, ?it/s]

provinces_territories: 13 features, invalid fixed pre/post-simplify=0/0


{'type': 'FeatureCollection',
 'features': [{'id': '0',
   'type': 'Feature',
   'properties': {'PRENAME': 'Alberta', 'PREABBR': 'Alta.', 'PRUID': '48'},
   'geometry': {'type': 'Polygon',
    'coordinates': [[[-110.0, 60.0],
      [-110.0, 56.28715],
      [-110.00581, 56.28216],
      [-110.0, 56.24256],
      [-110.00006, 55.33123],
      [-110.00754, 54.40663],
      [-110.00302, 50.83377],
      [-110.00802, 50.28004],
      [-110.00403, 49.95927],
      [-110.00479, 49.90452],
      [-110.01039, 49.87277],
      [-110.00421, 49.8144],
      [-110.00502, 48.9997],
      [-110.99315, 48.99787],
      [-111.60858, 48.99685],
      [-112.94325, 48.99846],
      [-113.66089, 48.99767],
      [-114.06833, 48.99885],
      [-114.07234, 49.00495],
      [-114.05376, 49.02654],
      [-114.0635, 49.04531],
      [-114.08098, 49.05969],
      [-114.09648, 49.05975],
      [-114.11581, 49.07382],
      [-114.12959, 49.0781],
      [-114.15315, 49.09951],
      [-114.14964, 49.11806],
      

## 7. Verify the output

Sanity-check every written file: valid JSON, a well-formed `FeatureCollection` (every feature
has both `geometry` and `properties`), no `NaN`/`Infinity` tokens, and that the single-city CSD
extracts (Section 3) have the same bounding box as their matching feature inside the full
collections (Section 4) - i.e. filtering-then-simplifying independently didn't silently pick up
the wrong polygon or distort it.


In [9]:
def bbox_of(fc, filt=None):
    xs, ys = [], []
    def walk(coords):
        if coords and isinstance(coords[0], (int, float)):
            xs.append(coords[0]); ys.append(coords[1])
        else:
            for c in coords:
                walk(c)
    for feat in fc["features"]:
        if filt and not filt(feat["properties"]):
            continue
        walk(feat["geometry"]["coordinates"])
    return (min(xs), min(ys), max(xs), max(ys))

json_files = sorted(OUTPUT_DIR.rglob("*.json"))
rows = []
loaded = {}
for path in tqdm(json_files, desc="verifying JSON"):
    raw = path.read_text()
    assert "NaN" not in raw and "Infinity" not in raw, f"{path} contains a non-JSON-safe token"
    data = json.loads(raw)  # raises if invalid JSON
    assert data["type"] == "FeatureCollection"
    for feat in data["features"]:
        assert "geometry" in feat and "properties" in feat
        assert feat["geometry"]["type"] in ("Polygon", "MultiPolygon")
    loaded[str(path.relative_to(OUTPUT_DIR))] = data
    rows.append({
        "file": str(path.relative_to(OUTPUT_DIR)),
        "features": len(data["features"]),
        "size_kb": round(len(raw) / 1024, 1),
    })

df = pd.DataFrame(rows)
print(f"{len(json_files)} files, all valid GeoJSON FeatureCollections, no NaN/Infinity tokens")
print(f"total size: {df['size_kb'].sum():.1f} KB")
df


verifying JSON:   0%|          | 0/9 [00:00<?, ?it/s]

9 files, all valid GeoJSON FeatureCollections, no NaN/Infinity tokens
total size: 689.4 KB


,file,features,size_kb
0,cma/toronto_hamilton.json,1,23.2
1,cma/vancouver.json,1,30.6
2,csd/gtha.json,27,72.0
3,csd/metro_vancouver.json,37,86.8
4,csd/toronto.json,1,25.5
5,csd/vancouver.json,1,69.2
6,provinces_territories.json,13,373.8
7,zone_2km/toronto.json,1,2.9
8,zone_2km/vancouver.json,1,5.4


In [10]:
# Toronto & Vancouver: single-city extract should closely match the corresponding feature
# inside the full CSD collection (same city, same underlying polygon). Both csd/toronto.json
# and csd/gtha.json are simplified at the same TOL_CSD tolerance, so those bounds should match
# exactly; csd/vancouver.json is intentionally left unsimplified (small already, see Section 3)
# while csd/metro_vancouver.json is simplified, so a ~1-unit-in-the-5th-decimal (~1m) tolerance
# is used for that comparison instead of exact equality.
def bboxes_close(a, b, tol=1e-4):
    return all(abs(x - y) <= tol for x, y in zip(a, b))

tor_bbox_single = bbox_of(loaded["csd/toronto.json"])
tor_bbox_in_gtha = bbox_of(loaded["csd/gtha.json"], lambda p: p["CSDNAME"] == "Toronto")
print("Toronto  - single extract bbox:", tor_bbox_single)
print("Toronto  - bbox within gtha.json:", tor_bbox_in_gtha)
assert tor_bbox_single == tor_bbox_in_gtha

van_bbox_single = bbox_of(loaded["csd/vancouver.json"])
van_bbox_in_metro = bbox_of(loaded["csd/metro_vancouver.json"], lambda p: p["CSDNAME"] == "Vancouver")
print("Vancouver - single extract bbox:", van_bbox_single)
print("Vancouver - bbox within metro_vancouver.json:", van_bbox_in_metro)
assert bboxes_close(van_bbox_single, van_bbox_in_metro)

print("bounding boxes match (within simplification tolerance) - both cities consistent "
      "across single-extract and full-collection files")


Toronto  - single extract bbox: (-79.6393, 43.5792, -79.11533, 43.85547)
Toronto  - bbox within gtha.json: (-79.6393, 43.5792, -79.11533, 43.85547)
Vancouver - single extract bbox: (-123.2249, 49.19986, -123.02294, 49.31408)
Vancouver - bbox within metro_vancouver.json: (-123.2249, 49.19987, -123.02294, 49.31408)
bounding boxes match (within simplification tolerance) - both cities consistent across single-extract and full-collection files
